# 07. Turn a runnable tutorial into a defensible research claim

A working pipeline is the beginning of the evidence, not the conclusion. This lesson designs controlled ablations, checks what has actually run, and connects the available evidence to the Med-WM submission scope.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Separate the proposed mechanisms

Uniform masking tests learning from missing joint-time observations. Positional noise changes the location representation. Stochastic latent variables allow multiple context-conditioned completions. VICReg terms constrain representation statistics. Changing all of these together would make an improvement hard to interpret.

The default model is a compact Skeleton-JEPA adaptation. The positional-noise switch is temporal jitter, not the full published StoP method. The stochastic latent extension borrows the prior/posterior idea but does not inherit image-paper performance guarantees.

In [ ]:
display(SVG(filename=str(PROJECT / "images/stochasticity.svg")))

In [ ]:
ablations = pd.DataFrame([
    {"identity":"deterministic", "position_jitter":0.0,"latent_dim":0,"question":"Reference predictive representation"},
    {"identity":"position-only", "position_jitter":0.2,"latent_dim":0,"question":"Does temporal position perturbation help?"},
    {"identity":"latent-only", "position_jitter":0.0,"latent_dim":4,"question":"Does prior diversity track future error?"},
    {"identity":"both", "position_jitter":0.2,"latent_dim":4,"question":"Do the two mechanisms interact?"},
])
display(ablations)

## Step 2. Declare the controls before testing

Use identical source folds, accepted windows, normal-pretraining data, masks, update budgets, and seed lists. Fit every fold's representation from scratch. Keep an untrained random-encoder probe to test whether useful structure already follows from architecture and pose input. Keep persistence for short-horizon prediction. Evaluate normal-only, cumulative adaptation, and mixed adaptation with separate identities.

Removing variance or covariance penalties is an anti-collapse ablation, not an instruction to seek maximally distant diagnosis clusters. Monitor dimension variance and effective representation rank alongside task performance.

In [ ]:
controls = pd.DataFrame([
    ["Random encoder", "Same architecture and training-only probe", "Representation architecture versus learned dynamics"],
    ["Persistence", "Same prefix, horizon, target mask, units", "Whether future learning beats a short-horizon rule"],
    ["Mixed adaptation", "Same parent and total updates", "Curriculum ordering, with exposure caveat"],
    ["Missingness/view audit", "Source-level nuisance metadata", "Acquisition patterns versus motion evidence"],
], columns=["Control","What stays matched","Why it matters"])
display(controls)

## Step 3. Make unfinished work explicit

A model configuration that instantiates successfully has not completed an ablation. A synthetic training run has not measured medical accuracy. A single real pilot has not established seed stability. Track these stages with distinct status fields so planned work is never presented as a result.

In [ ]:
if REAL:
    import json
    revision_development = Path(os.getenv("WM_REVISION_DEVELOPMENT") or PROJECT / "artifacts/runs/revision-development-v1")
    development_file = revision_development / "development.json"
    if not development_file.is_file():
        raise FileNotFoundError("Real notebook07 requires completed revision-development-v1/development.json; no synthetic ablation replacement is permitted.")
    revision_development_record = json.loads(development_file.read_text())
    report = revision_development_record
    normal_development_columns = ["seed", "arm", "normal_mse", "normal_energy", "persistence_gain", "normal_control_mse", "representation_diagnostics"]
    display(pd.DataFrame(revision_development_record["records"]).reindex(columns=normal_development_columns))
    print("Actual development evidence:", development_file)
    print("Validation gate:", revision_development_record.get("validation_gate"))
else:
    from worldmodels.model import ModelConfig, SkeletonJEPA
    rows = []
    for record in ablations.to_dict("records"):
        config = ModelConfig(dim=24, encoder_layers=1, position_jitter=record["position_jitter"], latent_dim=record["latent_dim"])
        model = SkeletonJEPA(config)
        rows.append({"identity":record["identity"], "parameters":sum(p.numel() for p in model.parameters() if p.requires_grad), "evidence":"instantiation only; no real ablation run"})
    display(pd.DataFrame(rows))

## Step 4. Inventory evidence for the paper

The workshop deadline and format must be checked against the live official call before submission. The locally verified call is documented below, including a date discrepancy between the website's update label and the workspace date. An honest position paper may be more appropriate than a strong empirical claim if the source review and controlled experiments are incomplete.

In [ ]:
display(Markdown((PROJECT / "docs/workshop.md").read_text()))

## Step 5. Read available results without assuming success

This cell lists completed local result records. An absent file is not a failed scientific result, but it is also not completed evidence. Keep the pilot's exact input identities, code hashes, versions, source support, and unresolved limitations in any manuscript supplement.

In [ ]:
import json
records = []
for path in sorted((PROJECT / "artifacts/runs").glob("*/results.json")):
    item = json.loads(path.read_text())
    records.append({"run":path.parent.name,"status":item.get("status"),"windows":item.get("n_windows"),"seed":item.get("seed"),"seconds":item.get("seconds")})
if records:
    display(pd.DataFrame(records))
else:
    print("No completed real result records were found.")

## Step 6. Write claims that the evidence can support

A reasonable initial claim concerns an auditable normal-reference prediction protocol and its behavior under source shifts. A claim that JEPA improves classification needs matched held-out predictions and uncertainty against RF and appropriate controls. A claim about uncertainty needs error-dispersion and calibration evidence. A claim of clinical generalization requires external patient or site validation.

Do not describe a residual heatmap as a causal neurological explanation. Do not describe projected coordinate motion as measured physical speed. Do not count overlapping windows as independent patients. Negative findings can be informative when the comparison is controlled and the limits are clear.

In [ ]:
claim_gate = pd.DataFrame([
    ["Runnable implementation", "Executed notebooks and focused tests", "Verify execution record"],
    ["Source-independent pilot", "Frozen sources, pose attrition, matched predictions", "Inspect pilot artifacts"],
    ["Stochasticity advantage", "Matched repeated-seed real-data ablations", "Not established by instantiation"],
    ["Clinical generalization", "External verified patient/site cohort", "Not established by this GAVD study"],
], columns=["Claim","Required evidence","Current interpretation"])
display(claim_gate)

## Step 7. Audit the executed revision and call its actual model

The real revision separates a direct coordinate predictor, a deterministic residual predictor, and a stochastic residual predictor. A residual predictor starts from a simple last-observation reference and learns corrections. Its contribution must be evaluated against persistence and the declared controls on matched sources. The development record identifies the actual settings, seeds, horizon, and validation evidence. The earlier four-row stochasticity table is a design map, not a claim that every listed option was executed.

This cell loads the saved deterministic residual model and calls it on an actual normal validation sequence from the extended cohort. The printed tensor dimensions verify the real forward path. Per-source CSVs provide the development errors used for the scientific comparison. Neither the input window nor these CSVs are drawn from the final test partition. The clean cohort reuses the original forecast checkpoints only because `docs/forecast_reuse_certificate.json` verifies exact equality of normal training, normal validation, and all test arrays, validity masks, and metadata. The source-score table is the filtered original validation prediction file, preserving its original stochastic draws. The forward call here is a pedagogical check and does not replace the inherited paper statistics. The 16-frame classification cohort and 32-frame forecasting cohort have different quality requirements and can retain different sources. Compare methods within their declared common cohort, and do not treat scores across those two populations as a paired comparison.

In [ ]:
if REAL:
    from worldmodels.revision import RevisionProtocol, load_revision_checkpoint, predict
    from worldmodels.workflow import load_prepared
    revision_input = Path(os.getenv("WM_REVISION_INPUT") or PROJECT / "artifacts/runs/extended-input-clean-v1")
    revision_x, revision_valid, revision_meta = load_prepared(revision_input)
    certificate_path = PROJECT / "docs/forecast_reuse_certificate.json"
    forecast_reuse_certificate = json.loads(certificate_path.read_text())
    required_checks = ("normal_train", "normal_validation", "all_test")
    assert all(forecast_reuse_certificate["checks"][name]["equal"] for name in required_checks)
    display(pd.DataFrame(forecast_reuse_certificate["checks"]).T)
    protocol = RevisionProtocol(**json.loads((revision_development / "protocol.json").read_text()))
    revision_model = load_revision_checkpoint(revision_development / "seed_27/residual_det.pt")
    validation_rows = revision_meta.index[(revision_meta.split == "validation") & (revision_meta.label == "normal")].to_numpy()
    if not len(validation_rows):
        raise ValueError("No normal validation windows in the real revision input")
    selected = validation_rows[:1]
    means, variances, samples = predict(revision_model, revision_x[selected], revision_valid[selected], protocol, seed=7028, control="actual")
    scores = {"revision_prediction": np.asarray(means), "revision_variance": np.asarray(variances), "revision_samples": np.asarray(samples)}
    print("Actual validation source:", revision_meta.loc[selected, "source_id"].tolist())
    print("Prediction shapes:", {name: value.shape for name, value in scores.items()})
    validation_source_scores = pd.read_csv(PROJECT / "artifacts/runs/revision-validation-clean-v1/revision-development-v1/seed_27/residual_det_actual_sources.csv")
    display(validation_source_scores.head())
    validation_source_scores.to_csv(lesson_directory("07_ablation_and_paper_evidence") / "revision_validation_sources.csv", index=False)
else:
    print("The real revision forward pass requires WM_DATA_MODE=real and completed development artifacts.")

### Secondary validation audit: coordinate-loss weight

Training-gradient diagnostics motivated a separately declared audit of the coordinate-loss weight, increasing it from 1 to 10 while retaining the same normal pretraining, source schedule, and three seeds. This is a secondary sensitivity analysis. The original weight-1 method remains the primary method regardless of whether the audit improves validation error. Its normal-source statistics are reused under the same clean-cohort equality certificate; filtered validation predictions retain their original lineage.

When its development artifact exists, the table reports actual normal-source error, the original method's error, persistence, and the predeclared context controls. It contains validation evidence only. An absent artifact is shown as pending and is never replaced with a synthetic result.

In [ ]:
objective_audit_path = Path(os.getenv("WM_OBJECTIVE_AUDIT") or PROJECT / "artifacts/runs/objective-audit-development-v1/development.json")
if REAL and objective_audit_path.is_file():
    objective_audit_report = json.loads(objective_audit_path.read_text())
    if objective_audit_report.get("declaration", {}).get("identity") != "coordinate-weight-ten-secondary-audit-v1":
        raise ValueError("Unexpected objective-audit identity")
    display(pd.DataFrame(objective_audit_report["records"]))
    print("Secondary validation audit only. The original weight-1 method remains primary.")
    objective_audit_output = lesson_directory("07_ablation_and_paper_evidence") / "objective_audit_validation.json"
    objective_audit_output.write_text(json.dumps(objective_audit_report, indent=2))
else:
    print("Secondary coordinate-weight audit is pending; no result has been substituted.")

## Step 8. Release final test evidence only after the protocol is frozen

The development evidence can motivate a declared method change. Once the method, primary endpoint, and analysis are fixed, the separate test command produces a new record. This notebook keeps the final test result behind `WM_RELEASE_TEST_RESULTS=1`. An unsuccessful validation gate is a result to report, not a reason to keep changing the method against the same final test outcomes.

In [ ]:
if REAL and RELEASE_TEST_RESULTS:
    revision_test = Path(os.getenv("WM_REVISION_TEST") or PROJECT / "artifacts/runs/revision-test-v1/results.json")
    if not revision_test.is_file():
        raise FileNotFoundError("Released real notebook07 requires a completed revision-test-v1/results.json")
    released_revision_test = json.loads(revision_test.read_text())
    display(released_revision_test)
    destination = lesson_directory("07_ablation_and_paper_evidence") / "released_revision_test.json"
    destination.write_text(json.dumps(released_revision_test, indent=2) + "\n")
else:
    print("Final revised test evidence remains sealed until WM_RELEASE_TEST_RESULTS=1.")

## Step 9. Inspect the consolidated evidence and figures

When the released analysis exists, this section reads its recorded interpretation and displays the evidence-linked figures. The binary comparison, forecast gain across seeds, and anomaly-score analysis answer different questions. A representation may support classification without beating persistence, and a prediction error may reflect pose quality rather than a medical condition. Read the uncertainty intervals and nuisance analyses before deciding which claims belong in a paper. The optional quality-only control is loaded from the separately released clean-cohort result, `quality-test-clean-v1/results.json`.

The teaching positional-jitter run in notebook 05 is a separate small demonstration. It should not be pooled with the three-seed revision or described as evidence of a stochasticity advantage.

In [ ]:
quality_test_path = Path(os.getenv("WM_QUALITY_TEST") or PROJECT / "artifacts/runs/quality-test-clean-v1/results.json")
if REAL and RELEASE_TEST_RESULTS and quality_test_path.is_file():
    quality_control_report = json.loads(quality_test_path.read_text())
    display(quality_control_report)
    quality_output = lesson_directory("07_ablation_and_paper_evidence") / "quality_control_test.json"
    quality_output.write_text(json.dumps(quality_control_report, indent=2))
research_path = PROJECT / "docs/research_results.json"
if REAL and RELEASE_TEST_RESULTS and research_path.is_file():
    research_report = json.loads(research_path.read_text())
    display(research_report.get("interpretation"))
    display({key: research_report[key] for key in ("forecast_validation_gate", "temporal_controls", "stochastic_energy_gain") if key in research_report})
    for figure_name in ("binary_revision.svg", "forecast_revision.svg", "anomaly_revision.svg"):
        figure_path = PROJECT / "images" / figure_name
        if figure_path.is_file():
            display(SVG(filename=str(figure_path)))
else:
    print("Consolidated research results are displayed after test release and analysis generation.")

In [ ]:
lesson_evidence = persist_lesson("07_ablation_and_paper_evidence", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.